# 05b. REINVENT4 LibInvent — prior-only sampling (Supplementary control)

**Purpose**: Descriptor-blind control for Supplementary comparison.

LibInvent prior sampling without RL provides a baseline to quantify
how much the task-directed RL (notebook 05) improves over blind scaffold decoration.

This is a **Supplementary** analysis only.
The primary benchmark uses task-directed RL (05_reinvent4_libinvent.ipynb).

Output: `results/new_benchmark/libinvent_prior_only.csv`


In [ ]:
import os, sys, subprocess, traceback
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import (
    ACTIVE_PARENT_DATASET, RESULTS_DIR,
    N_CANDIDATES_PER_PARENT, RANDOM_SEED, TASKS, TASK_NAMES,
)
from utils.parents import prepare_parent
from utils.scaffold import scaffold_is_retained
from utils.records import make_candidate_record
from utils.libinvent import make_libinvent_scaffold

os.makedirs(RESULTS_DIR, exist_ok=True)
PROJECT_ROOT = Path("..").resolve()

PRIOR_FILE        = PROJECT_ROOT / "external/reinvent4_priors/libinvent.prior"
MAX_ATTACHMENT_POINTS = 4
NUM_SMILES        = N_CANDIDATES_PER_PARENT
METHOD            = "libinvent_prior"
OUTPUT_FILE       = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent_prior_only.csv"
STATS_FILE        = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent_prior_only_generation_stats.csv"
INAPPLICABLE_FILE = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent_prior_only_inapplicable.csv"
ERROR_FILE        = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent_prior_only_errors.csv"


In [ ]:
df = pd.read_csv(ACTIVE_PARENT_DATASET)
if "canonical_smiles" not in df.columns and "smiles" in df.columns:
    df = df.rename(columns={"smiles": "canonical_smiles"})
if "parent_id" not in df.columns and "chembl_id" in df.columns:
    df = df.rename(columns={"chembl_id": "parent_id"})

scaffold_map = {}; inapplicable = []
for _, row in df.iterrows():
    prepared = prepare_parent(row)
    if prepared is None:
        inapplicable.append({"parent_id": row["parent_id"], "reason": "prepare_parent_failed"}); continue
    parent_id, parent_smiles, _, _ = prepared
    libinv_smi, reason = make_libinvent_scaffold(parent_smiles)
    if libinv_smi is None:
        inapplicable.append({"parent_id": parent_id, "reason": reason}); continue
    mol = Chem.MolFromSmiles(libinv_smi)
    n_ap = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 0)
    if n_ap > MAX_ATTACHMENT_POINTS:
        inapplicable.append({"parent_id": parent_id, "reason": "too_many_attachment_points"}); continue
    scaffold_map[parent_id] = (parent_smiles, libinv_smi)

print(f"Applicable: {len(scaffold_map)}  Inapplicable: {len(inapplicable)}")
pd.DataFrame(inapplicable).to_csv(INAPPLICABLE_FILE, index=False)


In [ ]:
# Prior-only: single sampling run (not task-dependent) → evaluate against all 7 tasks
WORK_DIR = PROJECT_ROOT / "results/libinvent_prior_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

parent_id_order = list(scaffold_map.keys())
input_smi = WORK_DIR / "scaffolds.smi"
raw_csv   = WORK_DIR / "raw_prior.csv"
config_path = PROJECT_ROOT / "configs/reinvent4_libinvent_sampling.toml"
log_path  = WORK_DIR / "log_prior.txt"

with open(input_smi, "w") as f:
    for pid in parent_id_order:
        f.write(scaffold_map[pid][1] + "\n")

toml_text = f'''
run_type = "sampling"
device = "cpu"

[parameters]
model_file = "{PRIOR_FILE}"
smiles_file = "{input_smi}"
output_file = "{raw_csv}"
num_smiles = {NUM_SMILES}
unique_molecules = true
randomize_smiles = true
'''.strip()
config_path.write_text(toml_text + "\n")

env = os.environ.copy(); env["KMP_DUPLICATE_LIB_OK"] = "TRUE"
result = subprocess.run(
    ["reinvent", "-l", str(log_path), "--seed", str(RANDOM_SEED), str(config_path)],
    capture_output=True, text=True, env=env,
)
if result.returncode != 0:
    raise RuntimeError(f"reinvent failed:\n{result.stderr[-2000:]}")
print("Done. Raw output:", raw_csv)


In [ ]:
df_raw = pd.read_csv(raw_csv)
SMILES_COL   = next((c for c in df_raw.columns if "smiles" in c.lower()), None)
SCAFFOLD_COL = next((c for c in df_raw.columns if "scaffold" in c.lower()
                     or "input" in c.lower() or "idx" in c.lower()), None)

records, gen_stats, error_rows = [], [], []

for sc_idx, parent_id in enumerate(parent_id_order):
    prepared = prepare_parent(pd.Series({
        "parent_id": parent_id, "canonical_smiles": scaffold_map[parent_id][0],
    }))
    if prepared is None: continue
    _, parent_smiles, parent_desc, _ = prepared
    parent_mol = Chem.MolFromSmiles(parent_smiles)

    if SCAFFOLD_COL:
        raw_smis = df_raw[df_raw[SCAFFOLD_COL] == sc_idx][SMILES_COL].dropna().tolist()
    else:
        start = sc_idx * NUM_SMILES
        raw_smis = df_raw[SMILES_COL].iloc[start:start + NUM_SMILES].dropna().tolist()

    n_attempted = len(raw_smis)
    n_invalid=0; n_scaffold_failed=0; n_duplicate=0
    candidates=[]; seen=set([parent_smiles])

    for smi in raw_smis:
        if len(candidates) >= NUM_SMILES: break
        cand_mol = Chem.MolFromSmiles(smi)
        if cand_mol is None: n_invalid+=1; continue
        canon = Chem.MolToSmiles(cand_mol)
        if not scaffold_is_retained(parent_mol, cand_mol): n_scaffold_failed+=1; continue
        if canon in seen: n_duplicate+=1; continue
        seen.add(canon); candidates.append(canon)

    n_valid_unique = len(candidates)
    assert n_invalid+n_scaffold_failed+n_duplicate+n_valid_unique == n_attempted
    gen_stats.append({"method": METHOD, "parent_id": parent_id,
                      "n_attempted": n_attempted, "n_invalid": n_invalid,
                      "n_scaffold_failed": n_scaffold_failed, "n_duplicate": n_duplicate,
                      "n_valid_unique": n_valid_unique})

    for task_name in TASK_NAMES:
        for rank, cand_smi in enumerate(candidates, start=1):
            rec = make_candidate_record(method=METHOD, parent_id=parent_id,
                parent_smiles=parent_smiles, task_name=task_name,
                candidate_rank=rank, candidate_smiles=cand_smi, parent_desc=parent_desc)
            if rec: records.append(rec)

print(f"Total records: {len(records)}")
pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
pd.DataFrame(gen_stats).to_csv(STATS_FILE, index=False)
pd.DataFrame(error_rows).to_csv(ERROR_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
